# Third-Generation CALPHAD: Einstein Oscillator

This example demonstrates how to use the `GroundStateNode` and `EinsteinNode` to build a physically consistent thermodynamic model for a pure phase that obeys the Third Law of Thermodynamics ($C_p \to 0$ as $T \to 0$ K). We construct pure Copper (Cu) and Aluminum (Al) phases and use `jax.grad` to extract their Heat Capacities.

In [ ]:
import os
import sys
sys.path.append(os.path.abspath('../../zgraph/src'))
sys.path.append(os.path.abspath('../../thermograph/src'))
sys.path.append(os.path.abspath('../../'))

import jax
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go

from zgraph import *
from thermograph.nodes.einstein import GroundStateNode, EinsteinNode
from external_data.einstein_params import CU_FCC, AL_FCC

## 1. Defining the ZGraph Architecture
We declare the independent variable $T$ and instantiate our purely normalized nodes.

In [ ]:
T = SignalNodes(0)

# Ground State Energies
cu_e0 = GroundStateNode(CU_FCC["E_0"]).compile_zgraph_engine()
al_e0 = GroundStateNode(AL_FCC["E_0"]).compile_zgraph_engine()

# 1-DOF Einstein Oscillators
cu_osc = EinsteinNode(CU_FCC["theta_acoustic"], T_index=0).compile_zgraph_engine()
al_osc = EinsteinNode(AL_FCC["theta_acoustic"], T_index=0).compile_zgraph_engine()


## 2. PGM Topology: Connecting the Leaves
We use a `FactorNode` to apply the structural weights. A pure monatomic lattice has 3 degrees of freedom, so we scale the 1-DOF oscillator by a factor of 3 in the connection matrix.

In [ ]:
# The weight matrix M applies [1.0 * E_0, 3.0 * Oscillator]
cu_phase = FactorNode(jnp.array([[1.0, 3.0]]), [cu_e0, cu_osc], beta=0.0)
al_phase = FactorNode(jnp.array([[1.0, 3.0]]), [al_e0, al_osc], beta=0.0)



## 3. Deriving Heat Capacity
Using JAX, we can automatically derive the heat capacity $C_p = -T \frac{\partial^2 G}{\partial T^2}$. Because the graph is fully differentiable across all temperature ranges, we can evaluate it directly down to absolute zero.

In [ ]:
# Compute dG/dT (Entropy is -dG/dT)
cu_grad = jax.grad(lambda t: cu_phase(jnp.atleast_1d(t)))
al_grad = jax.grad(lambda t: al_phase(jnp.atleast_1d(t)))

# Compute d2G/dT2
cu_grad2 = jax.grad(cu_grad)
al_grad2 = jax.grad(al_grad)

# Vectorize the functions for batch evaluation
cu_cp = jax.vmap(lambda t: -t * cu_grad2(t))
al_cp = jax.vmap(lambda t: -t * al_grad2(t))

T_vals = jnp.linspace(0.1, 1000, 500)
cp_cu_vals = cu_cp(T_vals)
cp_al_vals = al_cp(T_vals)


## 4. Visualizing Physical Consistency
Plotting the Heat Capacity reveals the hallmark signature of the Einstein model: unlike polynomial fits which often incorrectly predict finite or negative heat capacities at 0 K, the oscillator model smoothly and correctly forces $C_p \to 0$ as $T \to 0$.

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=T_vals, y=cp_cu_vals, mode='lines', name='Cu (FCC)', line=dict(color='orange', width=3)))
fig.add_trace(go.Scatter(x=T_vals, y=cp_al_vals, mode='lines', name='Al (FCC)', line=dict(color='blue', width=3)))

fig.update_layout(
    title='Heat Capacity (Einstein Model)',
    xaxis_title='Temperature (K)',
    yaxis_title='Cp (J / mol K)',
    width=800,
    height=500
)
fig.show()


## 5. Constructing a Hybrid Phase Diagram
We can seamlessly mix our physically consistent `EinsteinNode` solid phases with standard `SGTENode` liquid phases to create a complete Cu-Al Phase Diagram!

In [ ]:
from thermograph.nodes.sgte import SGTENode
from external_data.cu_ni_sgte import GLIQCU

# Manually extracted GLIQAL
GLIQAL = [
    (933.473, [11005.045-11276.24, -11.84185+223.048446, -38.5844296, 18.531982e-3, 74092, -5.764227e-6, 79.337e-21, 0]),
    (2900.0, [-795.991, 177.430209, -31.748192, 0, 0, 0, 0, 0])
]

cu_liq_node = SGTENode(GLIQCU, T_index=0).compile_zgraph_engine()
al_liq_node = SGTENode(GLIQAL, T_index=0).compile_zgraph_engine()

T, mu_Cu, mu_Al = SignalNodes(0, 1, 2)
RT = FactorNode([[8.314]], [T])

# Mix the Einstein Solid Phases
w_cu_fcc = FactorNode([[1.0, -1.0]], [mu_Cu, cu_phase])
w_al_fcc = FactorNode([[1.0, -1.0]], [mu_Al, al_phase])
phase_FCC = FactorNode(jnp.eye(2), [w_cu_fcc, w_al_fcc], beta=RT)

# Mix the SGTE Liquid Phases
w_cu_liq = FactorNode([[1.0, -1.0]], [mu_Cu, cu_liq_node])
w_al_liq = FactorNode([[1.0, -1.0]], [mu_Al, al_liq_node])
phase_LIQ = FactorNode(jnp.eye(2), [w_cu_liq, w_al_liq], beta=RT)

system = FactorNode(jnp.eye(2), [phase_FCC, phase_LIQ], beta=0.0)


## 6. Extract Boundaries
We use the `PhaseBoundaryPredictor` to compute the solidus and liquidus lines.

In [ ]:
from thermograph.prediction import PhaseBoundaryPredictor

predictor = PhaseBoundaryPredictor(system)
T_vals_pd = jnp.linspace(800, 1400, 100)
mu_diff = jnp.linspace(-80000, 80000, 200)
T_grid, mu_grid = jnp.meshgrid(T_vals_pd, mu_diff, indexing='ij')
inputs = jnp.stack([T_grid, mu_grid/2, -mu_grid/2], axis=-1)

batched_x = predictor.predict_compositions(inputs, mu_index=2)
x_left = jnp.minimum(batched_x[:, 0], batched_x[:, 1])
x_right = jnp.maximum(batched_x[:, 0], batched_x[:, 1])


In [ ]:
fig_pd = go.Figure()

T_np = np.asarray(T_vals_pd)
xl_np = np.asarray(x_left)
xr_np = np.asarray(x_right)

fig_pd.add_trace(go.Scatter(x=xl_np, y=T_np, mode='lines', name='Solidus (Einstein)', line=dict(color='red', width=3)))
fig_pd.add_trace(go.Scatter(x=xr_np, y=T_np, mode='lines', name='Liquidus (SGTE)', line=dict(color='blue', width=3)))

fig_pd.update_layout(
    title='Hybrid Cu-Al Phase Diagram (Einstein Solid + SGTE Liquid)',
    xaxis_title='Mole Fraction Al',
    yaxis_title='Temperature (K)',
    width=800,
    height=600
)
fig_pd.show()
